# Submissão 2 - Abordagem 1: Fine-Tuning de Transformer (DistilBERT) Hierárquico

Neste notebook implementamos uma pipeline hierárquica tal e qual à desenhada para a Regressão Logística:
1. **Modelo Binário**: Distingue textos Humanos vs Textos Gerados por IA.
2. **Modelo Multiclasse**: Para os textos previstos como IA, prevê qual foi a origem (Google, Meta, Mistral, OpenAI).

In [ ]:

!pip install transformers -q
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, accuracy_score
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from torch.optim import AdamW
import os

DATA_DIR = "../data"
MODELS_DIR = "../models"
os.makedirs(MODELS_DIR, exist_ok=True)

df_train = pd.read_csv(os.path.join(DATA_DIR, "dataset_treino.csv"))
df_val = pd.read_csv(os.path.join(DATA_DIR, "dataset_validacao.csv"))

def clean_text(text):
    import re
    text = str(text).replace("\n", " ").replace("\r", " ")
    return re.sub(r"\s+", " ", text).strip()

for df in [df_train, df_val]:
    df['Text'] = df['Text'].apply(clean_text)

unique_labels = df_train['source_name'].dropna().unique().tolist()
mapping = {k: v for v, k in enumerate(sorted(unique_labels))}
reverse_mapping = {v: k for k, v in mapping.items()}
print("Mapeamento Original:", mapping)

df_train['binary_label'] = df_train['source_name'].apply(lambda x: 0 if x.lower() == 'human' else 1)
df_val['binary_label'] = df_val['source_name'].apply(lambda x: 0 if x.lower() == 'human' else 1)

df_train_ai = df_train[df_train['binary_label'] == 1].copy()
df_val_ai = df_val[df_val['binary_label'] == 1].copy()

ai_classes = [c for c in unique_labels if c.lower() != 'human']
ai_mapping = {k: v for v, k in enumerate(sorted(ai_classes))}
reverse_ai_mapping = {v: k for k, v in ai_mapping.items()}
print("Mapeamento Apenas IA:", ai_mapping)

df_train_ai['ai_label'] = df_train_ai['source_name'].map(ai_mapping)
df_val_ai['ai_label'] = df_val_ai['source_name'].map(ai_mapping)


In [ ]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=max_length)
        self.labels = labels.tolist()
    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item['labels'] = torch.tensor(self.labels[idx])
        return item
    def __len__(self):
        return len(self.labels)

train_bin_dataset = TextDataset(df_train['Text'], df_train['binary_label'], tokenizer)
val_bin_dataset = TextDataset(df_val['Text'], df_val['binary_label'], tokenizer)

train_bin_loader = DataLoader(train_bin_dataset, batch_size=32, shuffle=True)
val_bin_loader = DataLoader(val_bin_dataset, batch_size=64)

binary_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=2)
binary_model.to(device)
optim_bin = AdamW(binary_model.parameters(), lr=5e-5)

print("A treinar Modelo Binário ")
for epoch in range(2): 
    binary_model.train()
    for batch in train_bin_loader:
        optim_bin.zero_grad()
        outputs = binary_model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device), labels=batch['labels'].to(device))
        loss = outputs.loss
        loss.backward()
        optim_bin.step()
    print(f"Época {epoch+1} - Loss Binária: {loss.item():.4f}")

binary_model.save_pretrained(os.path.join(MODELS_DIR, 'bert_binary_model'))


In [ ]:

train_multi_dataset = TextDataset(df_train_ai['Text'], df_train_ai['ai_label'], tokenizer)
val_multi_dataset = TextDataset(df_val_ai['Text'], df_val_ai['ai_label'], tokenizer)

train_multi_loader = DataLoader(train_multi_dataset, batch_size=32, shuffle=True)
val_multi_loader = DataLoader(val_multi_dataset, batch_size=64)

multi_model = DistilBertForSequenceClassification.from_pretrained('distilbert-base-uncased', num_labels=len(ai_classes))
multi_model.to(device)
optim_multi = AdamW(multi_model.parameters(), lr=5e-5)

print("\nA treinar Modelo Multiclasse (Qual IA?)...")
for epoch in range(3):
    multi_model.train()
    for batch in train_multi_loader:
        optim_multi.zero_grad()
        outputs = multi_model(batch['input_ids'].to(device), attention_mask=batch['attention_mask'].to(device), labels=batch['labels'].to(device))
        loss = outputs.loss
        loss.backward()
        optim_multi.step()
    print(f"Época {epoch+1} - Loss Multiclasse: {loss.item():.4f}")

multi_model.save_pretrained(os.path.join(MODELS_DIR, 'bert_multi_model'))


In [ ]:

def hierarchical_inference(texts, tokenizer, binary_model, multi_model):
    binary_model.eval()
    multi_model.eval()
    
    predictions = []
    
    batch_size = 64
    for i in range(0, len(texts), batch_size):
        batch_texts = texts[i:i+batch_size].tolist()
        encodings = tokenizer(batch_texts, truncation=True, padding=True, max_length=128, return_tensors='pt').to(device)
        
        with torch.no_grad():
            bin_outputs = binary_model(**encodings)
            bin_preds = torch.argmax(bin_outputs.logits, dim=1).cpu().numpy()
            
        for idx, is_ai in enumerate(bin_preds):
            if is_ai == 0:
                predictions.append('human')
            else:
                single_enc = {k: v[idx].unsqueeze(0) for k, v in encodings.items()}
                with torch.no_grad():
                    multi_out = multi_model(**single_enc)
                    multi_pred = torch.argmax(multi_out.logits, dim=1).cpu().item()
                    predictions.append(reverse_ai_mapping[multi_pred])
                    
    return predictions

print("previsões hierárquicas no dataset de validação...")
val_preds = hierarchical_inference(df_val['Text'], tokenizer, binary_model, multi_model)


print(classification_report(df_val['source_name'], val_preds))
print("Accuracy Final:", accuracy_score(df_val['source_name'], val_preds))


df_submission = df_val.copy()
df_submission['predicted_source'] = val_preds
csv_out = '../subm2/subm_bert_hierarquico.csv'
os.makedirs('../subm2', exist_ok=True)
df_submission.to_csv(csv_out, index=False)
print(f"\nSubmissão gerada em {csv_out}")
